
## Perceptron Learning Rule with Different Activation Functions
**Dataset:** Breast Cancer Wisconsin (Binary Classification)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 110

In [ ]:
data = load_breast_cancer()
df   = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target          # 1 = benign, 0 = malignant

print("Shape:", df.shape)
print("\nClass distribution:")
print(df['target'].value_counts().rename({1:'Benign', 0:'Malignant'}))
print("\nMissing values:", df.isnull().sum().sum())
df.describe().T[['mean','std','min','max']].head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution
df['target'].value_counts().rename({1:'Benign', 0:'Malignant'}).plot(
    kind='bar', ax=axes[0], color=['steelblue','tomato'], edgecolor='black')
axes[0].set_title('Class Distribution'); axes[0].set_xlabel('')

# Feature correlation heatmap (first 10 features)
sns.heatmap(df.iloc[:, :10].corr(), ax=axes[1], cmap='coolwarm',
            annot=False, linewidths=0.3)
axes[1].set_title('Feature Correlation (first 10)')
plt.tight_layout(); plt.show()

In [ ]:
X = df.drop('target', axis=1).values
y = df['target'].values

# Split: 70% train | 15% val | 15% test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30,
                                                    random_state=42, stratify=y)
X_val, X_test, y_val, y_test     = train_test_split(X_temp, y_temp, test_size=0.50,
                                                    random_state=42, stratify=y_temp)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print(f"Train: {X_train.shape}  |  Val: {X_val.shape}  |  Test: {X_test.shape}")

In [ ]:
# ── CELL 5: Activation Functions ─────────────────────────────────────────────
def binary_step(z):  return np.where(z >= 0, 1, 0)
def sign_fn(z):      return np.where(z >= 0, 1, -1)
def sigmoid(z):      return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
def tanh_fn(z):      return np.tanh(z)

# Threshold functions (map continuous output → {0,1})
def threshold(output, fn_name):
    if fn_name == 'sign':    return np.where(output >= 0, 1, 0)
    if fn_name == 'tanh':    return np.where(output >= 0, 1, 0)
    return np.where(output >= 0.5, 1, 0)   # sigmoid

ACTIVATIONS = {
    'binary_step': binary_step,
    'sign':        sign_fn,
    'sigmoid':     sigmoid,
    'tanh':        tanh_fn,
}

In [ ]:
class Perceptron:
    def __init__(self, activation_fn, activation_name,
                 lr=0.01, max_epochs=1000, seed=42):
        self.fn      = activation_fn
        self.fn_name = activation_name
        self.lr      = lr
        self.max_ep  = max_epochs
        self.seed    = seed

    def _predict_raw(self, X):
        return self.fn(X @ self.w + self.b)

    def _to_binary(self, raw):
        if self.fn_name in ('binary_step',):
            return raw.astype(int)
        return threshold(raw, self.fn_name)

    def fit(self, X, y):
        rng        = np.random.RandomState(self.seed)
        n, d       = X.shape
        self.w     = rng.uniform(-0.5, 0.5, d)
        self.b     = 0.0
        self.errors_per_epoch = []
        self.conv_epoch       = self.max_ep

        for epoch in range(1, self.max_ep + 1):
            errors = 0
            for xi, yi in zip(X, y):
                raw  = self.fn(np.dot(xi, self.w) + self.b)
                pred = int(self._to_binary(np.array([raw]))[0])
                err  = yi - pred
                if err != 0:
                    self.w += self.lr * err * xi
                    self.b += self.lr * err
                    errors += 1
            self.errors_per_epoch.append(errors)
            if errors == 0:
                self.conv_epoch = epoch
                break
        return self

    def predict(self, X):
        return self._to_binary(self._predict_raw(X))

In [ ]:
results   = {}
models    = {}

for name, fn in ACTIVATIONS.items():
    p = Perceptron(fn, name, lr=0.01, max_epochs=1000).fit(X_train, y_train)
    y_pred = p.predict(X_test)

    results[name] = {
        'accuracy':   accuracy_score(y_test, y_pred),
        'precision':  precision_score(y_test, y_pred, zero_division=0),
        'recall':     recall_score(y_test, y_pred, zero_division=0),
        'f1':         f1_score(y_test, y_pred, zero_division=0),
        'conv_epoch': p.conv_epoch,
        'errors':     p.errors_per_epoch,
        'cm':         confusion_matrix(y_test, y_pred),
    }
    models[name] = p
    print(f"{name:<15} | Acc: {results[name]['accuracy']:.4f} | "
          f"F1: {results[name]['f1']:.4f} | Converged @ epoch {p.conv_epoch}")

In [ ]:
metrics_df = pd.DataFrame({
    name: {
        'Accuracy':    f"{v['accuracy']:.4f}",
        'Precision':   f"{v['precision']:.4f}",
        'Recall':      f"{v['recall']:.4f}",
        'F1-Score':    f"{v['f1']:.4f}",
        'Conv. Epoch': v['conv_epoch'],
    } for name, v in results.items()
}).T
print(metrics_df.to_string())

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, v) in zip(axes, results.items()):
    sns.heatmap(v['cm'], annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Mal','Ben'], yticklabels=['Mal','Ben'])
    ax.set_title(f"{name}\nAcc={v['accuracy']:.3f}")
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.suptitle('Confusion Matrices – All Activation Functions', y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
for name, v in results.items():
    plt.plot(v['errors'], label=name)
plt.xlabel('Epoch'); plt.ylabel('Misclassifications')
plt.title('Training Error vs Epochs')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

names  = list(results.keys())
accs   = [results[n]['accuracy']  for n in names]
f1s    = [results[n]['f1']        for n in names]
colors = ['steelblue','tomato','seagreen','darkorange']

axes[0].bar(names, accs, color=colors, edgecolor='black')
axes[0].set_ylim(0, 1.1)
axes[0].set_title('Accuracy Comparison')
for i, v in enumerate(accs): axes[0].text(i, v+0.01, f"{v:.3f}", ha='center')

axes[1].bar(names, f1s, color=colors, edgecolor='black')
axes[1].set_ylim(0, 1.1)
axes[1].set_title('F1-Score Comparison')
for i, v in enumerate(f1s): axes[1].text(i, v+0.01, f"{v:.3f}", ha='center')

plt.tight_layout(); plt.show()

In [ ]:
from sklearn.decomposition import PCA

pca      = PCA(n_components=2, random_state=42)
X_tr_2d  = pca.fit_transform(X_train)
X_te_2d  = pca.transform(X_test)

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, (name, fn) in zip(axes, ACTIVATIONS.items()):
    p2 = Perceptron(fn, name, lr=0.01, max_epochs=1000).fit(X_tr_2d, y_train)

    x_min, x_max = X_te_2d[:,0].min()-1, X_te_2d[:,0].max()+1
    y_min, y_max = X_te_2d[:,1].min()-1, X_te_2d[:,1].max()+1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    grid   = np.c_[xx.ravel(), yy.ravel()]
    Z      = p2.predict(grid).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    for cls, clr, lbl in [(0,'tomato','Malignant'),(1,'steelblue','Benign')]:
        mask = y_test == cls
        ax.scatter(X_te_2d[mask,0], X_te_2d[mask,1],
                   c=clr, label=lbl, edgecolor='k', s=25, alpha=0.8)
    ax.set_title(f"{name}\nAcc={accuracy_score(y_test, p2.predict(X_te_2d)):.3f}")
    ax.legend(fontsize=7); ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

plt.suptitle('Decision Boundaries (PCA 2D)', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
lrs      = [0.001, 0.01, 0.05, 0.1, 0.5]
best_fn  = 'sigmoid'   # change if your results differ
lr_accs  = []

for lr in lrs:
    p = Perceptron(ACTIVATIONS[best_fn], best_fn, lr=lr, max_epochs=1000).fit(X_train, y_train)
    lr_accs.append(accuracy_score(y_test, p.predict(X_test)))

plt.figure(figsize=(7,4))
plt.plot(lrs, lr_accs, 'o-', color='steelblue')
plt.xscale('log'); plt.xlabel('Learning Rate (log scale)')
plt.ylabel('Test Accuracy'); plt.title(f'LR vs Accuracy ({best_fn})')
plt.grid(True, alpha=0.4); plt.tight_layout(); plt.show()

for lr, acc in zip(lrs, lr_accs):
    print(f"LR={lr:.3f}  →  Acc={acc:.4f}")

In [ ]:
epoch_list = [10, 50, 100, 300, 500, 1000]
ep_accs    = []

for ep in epoch_list:
    p = Perceptron(ACTIVATIONS[best_fn], best_fn, lr=0.01, max_epochs=ep).fit(X_train, y_train)
    ep_accs.append(accuracy_score(y_test, p.predict(X_test)))

plt.figure(figsize=(7,4))
plt.plot(epoch_list, ep_accs, 's-', color='seagreen')
plt.xlabel('Max Epochs'); plt.ylabel('Test Accuracy')
plt.title(f'Max Epochs vs Accuracy ({best_fn})')
plt.grid(True, alpha=0.4); plt.tight_layout(); plt.show()

In [ ]:
seeds    = [0, 7, 13, 42, 99]
seed_accs = {name: [] for name in ACTIVATIONS}

for seed in seeds:
    for name, fn in ACTIVATIONS.items():
        p = Perceptron(fn, name, lr=0.01, max_epochs=1000, seed=seed).fit(X_train, y_train)
        seed_accs[name].append(accuracy_score(y_test, p.predict(X_test)))

plt.figure(figsize=(9,4))
for name, accs in seed_accs.items():
    plt.plot(seeds, accs, 'o-', label=name)
plt.xlabel('Random Seed (proxy for initial weights)')
plt.ylabel('Test Accuracy'); plt.title('Sensitivity to Initial Weights')
plt.legend(); plt.grid(True, alpha=0.4); plt.tight_layout(); plt.show()

In [ ]:
print("=" * 60)
print("         FINAL PERFORMANCE SUMMARY")
print("=" * 60)
print(f"{'Activation':<15} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7} {'Conv@':>7}")
print("-" * 60)
for name, v in results.items():
    print(f"{name:<15} {v['accuracy']:>7.4f} {v['precision']:>7.4f} "
          f"{v['recall']:>7.4f} {v['f1']:>7.4f} {v['conv_epoch']:>7}")
print("=" * 60)

best = max(results, key=lambda k: results[k]['f1'])
print(f"\n✅ Best Activation Function: {best.upper()}")
print(f"   F1-Score: {results[best]['f1']:.4f}  |  Accuracy: {results[best]['accuracy']:.4f}")

## Task 6: Observations

### 1. Effect of Different Activation Functions
- **Binary Step & Sign** produce hard binary decisions. They converge faster on linearly separable subsets but offer no probabilistic output and are sensitive to noise.
- **Sigmoid** outputs values in (0,1), providing a smooth gradient-like update signal even in a perceptron-style rule, which tends to give more stable convergence.
- **Tanh** maps outputs to (−1,1); it is zero-centred which can yield slightly faster convergence than sigmoid in practice.

### 2. Convergence Behaviour
- The Perceptron Learning Rule **guarantees convergence only** if the data is linearly separable.
- On the Breast Cancer dataset (not perfectly linearly separable in raw space), step/sign functions may oscillate; sigmoid and tanh are more robust because their smooth output dampens oscillation.

### 3. Advantages & Limitations
| | |
|---|---|
| ✅ Simple, fast, interpretable | ❌ Only works on linearly separable data |
| ✅ Low memory footprint | ❌ No hidden representation learning |
| ✅ Online learning ready | ❌ No probabilistic output (step/sign) |

### 4. Why Only Linearly Separable?
A single perceptron computes a **linear decision boundary** (hyperplane). For datasets with non-linear class boundaries (e.g. XOR, concentric circles), this hyperplane cannot separate the classes, and the learning rule never converges.

### 5. When to Prefer MLP over Perceptron
- Non-linearly separable data (XOR, spirals)
- Multi-class problems
- High-dimensional image or text tasks
- When feature interactions and hierarchical representations matter

### Conclusion
The **sigmoid / tanh** activation functions outperformed binary step and sign on the Breast Cancer dataset due to smoother weight updates. The perceptron is a foundational model but is superseded by MLPs for any real-world non-linear problem.